# Análisis LiDAR con Python y PDAL
## Comparativa PNOA 2024 vs vuelo dron Zenmuse L1 — Alcolea del Río

---

En este notebook vamos a aprender a trabajar con nubes de puntos LiDAR usando **PDAL** (_Point Data Abstraction Library_), una de las bibliotecas más potentes para procesamiento de datos de nube de puntos.

### Datos que usamos

| Fuente | Sensor | Fecha | Descripción |
|--------|--------|-------|-------------|
| **PNOA-LiDAR 3ª cobertura** | Avión | 2024 | 11 tiles 1×1 km, clasificados, ETRS89/UTM30N, altura ortométrica |
| **Vuelo dron** | DJI Zenmuse L1 | 2024 | Un vuelo sobre Alcolea del Río, RTK con base D-RTK 2 autoposicionada |

### Lo que aprenderemos

1. Mosaicar tiles del PNOA con `laspy`
2. Arquitectura de PDAL: pipelines de lectura → filtros → escritura
3. Inspeccionar nubes de puntos: metadatos, bounds, clasificaciones
4. Problema habitual en drones DJI: CRS no escrito y alturas elipsoidales
5. Recortar una nube al área de otra (por bbox y por hull real)
6. Generar DTM, DSM y CHM con PDAL
7. Comparar los modelos de ambas fuentes y cuantificar el sesgo Z

## 0. Setup: imports y rutas

Centralizamos todas las rutas aquí para facilitar la adaptación a otros proyectos.

In [ ]:
import pdal
import json
import laspy
import numpy as np
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import geopandas as gpd
from shapely import wkt
from pathlib import Path
import time

print(f'PDAL version: {pdal.__version__}')
print(f'laspy version: {laspy.__version__}')

In [ ]:
# ── RUTAS DE TRABAJO ─────────────────────────────────────────────────────────
# Adapta WORKDIR a tu máquina

WORKDIR    = Path('/home/diego/Descargas')
TILES_DIR  = Path('/media/diego/Datos4/EBD/Tragsatec/Drones/Laz/las12/alcolea')

# ── Archivos de entrada ───────────────────────────────────────────────────────
L1_ORIG    = WORKDIR / 'cloud92048a649bb1c353.copc.laz'   # L1 original (sin CRS, elipsoidal)
L1_ORTOM   = WORKDIR / 'l1_ortometrica.copc.laz'          # L1 reproyectado (ETRS89 + orth.)
PNOA_MOS   = WORKDIR / 'merged_11tiles.copc.laz'          # Mosaico PNOA (11 tiles)
PNOA_CROP  = WORKDIR / 'pnoa_recortado.copc.laz'          # PNOA recortado al extent del L1

# ── Productos raster ─────────────────────────────────────────────────────────
DTM_L1     = WORKDIR / 'dtm_l1.tif'
DSM_L1     = WORKDIR / 'dsm_l1.tif'
CHM_L1     = WORKDIR / 'chm_l1.tif'
DTM_PNOA   = WORKDIR / 'dtm_pnoa.tif'
DSM_PNOA   = WORKDIR / 'dsm_pnoa.tif'
CHM_PNOA   = WORKDIR / 'chm_pnoa.tif'

# ── Verificar existencia ──────────────────────────────────────────────────────
for nombre, ruta in [('L1 original', L1_ORIG), ('L1 ortométrica', L1_ORTOM),
                     ('PNOA mosaico', PNOA_MOS), ('PNOA recortado', PNOA_CROP)]:
    estado = '✓' if ruta.exists() else '✗ NO ENCONTRADO'
    print(f'  {estado}  {nombre}: {ruta.name}')

---
## 1. Mosaico de tiles PNOA con `laspy`

### ¿Por qué necesitamos mosaicarlo?

El PNOA-LiDAR se distribuye en **tiles de 1×1 km** (o 2×2 km en algunas coberturas). Para trabajar con el área completa del vuelo dron necesitamos unir varios tiles en un solo archivo.

El área de Alcolea del Río requiere **11 tiles** PNOA 2024.

> **Nota:** Este paso ya fue ejecutado en el módulo de LiDAR del curso GeoPython 2026.  
> El resultado (`merged_11tiles.copc.laz`) ya existe. Revisamos el código para entender el proceso.

In [ ]:
# Listar los tiles PNOA disponibles
if TILES_DIR.exists():
    tiles = sorted(TILES_DIR.glob('PNOA_2024_AND_*.copc.laz'))
    print(f'Tiles encontrados: {len(tiles)}')
    for t in tiles:
        size_mb = t.stat().st_size / 1024**2
        print(f'  {t.name}  ({size_mb:.0f} MB)')
else:
    print('Directorio de tiles no disponible en esta máquina')
    print('El mosaico ya está generado en:', PNOA_MOS)

In [ ]:
# ── CÓDIGO DE MOSAICADO (ya ejecutado, no re-ejecutar) ────────────────────────
#
# Estrategia: usar laspy.LasWriter para escribir todos los tiles uno a uno
# sin cargarlos todos en memoria simultáneamente.
# Esto es importante con 11 tiles × ~10 M puntos cada uno = ~110 M puntos totales.

def mosaico_pnoa(tiles_dir, salida, patron='PNOA_2024_AND_*.copc.laz'):
    """
    Une todos los tiles LAZ de un directorio en un solo archivo.
    Usa escritura incremental para no saturar la RAM.
    """
    archivos = sorted(Path(tiles_dir).glob(patron))
    print(f'{len(archivos)} tiles encontrados')

    # Leer todos los headers para verificar compatibilidad
    las_list = []
    for f in archivos:
        las_list.append(laspy.read(f))
        print(f'  Leído: {f.name}  ({len(las_list[-1].points):,} pts)')

    total_pts = sum(len(l.points) for l in las_list)
    print(f'\nTotal: {total_pts:,} puntos')

    # Construir header con los parámetros del primer tile
    header = laspy.LasHeader(
        point_format=las_list[0].header.point_format,
        version=las_list[0].header.version
    )
    header.offsets = las_list[0].header.offsets
    header.scales  = las_list[0].header.scales

    print(f'\nEscribiendo mosaico en {salida}...')
    with open(salida, 'wb') as f:
        with laspy.LasWriter(f, header=header) as writer:
            for i, las in enumerate(las_list):
                writer.write_points(las.points)
                print(f'  Tile {i+1}/{len(las_list)} escrito')
    print('Mosaico completado.')

# Para ejecutar:
# mosaico_pnoa(TILES_DIR, WORKDIR / 'merged_11tiles.laz')
print('Función definida. El mosaico ya existe en:', PNOA_MOS.name)

---
## 2. Arquitectura de PDAL

PDAL funciona mediante **pipelines**: secuencias de operaciones definidas en JSON que procesan la nube de puntos en flujo continuo.

```
READER  →  FILTER  →  FILTER  →  WRITER
  │                                 │
 Lee                              Escribe
 datos                            resultado
```

Ventajas de este diseño:
- **Encadenado eficiente**: los puntos fluyen uno a uno, sin cargar todo en RAM
- **Reproducibilidad**: el pipeline JSON es documentación ejecutable
- **Flexibilidad**: se pueden combinar operaciones libremente

### Tipos de operaciones

| Tipo | Prefijo | Ejemplos |
|------|---------|----------|
| Lectores | `readers.` | `readers.copc`, `readers.las`, `readers.e57` |
| Filtros | `filters.` | `filters.crop`, `filters.range`, `filters.reprojection`, `filters.hexbin` |
| Escritores | `writers.` | `writers.copc`, `writers.las`, `writers.gdal` |

### COPC vs LAS/LAZ

**COPC** (_Cloud Optimized Point Cloud_) es un formato LAZ con un índice octree interno.  
Permite **lectura parcial** (por bbox, polígono o resolución) sin leer el archivo completo: ideal para archivos grandes.

```python
# Leer solo un área y baja resolución → milisegundos en vez de minutos
{"type": "readers.copc", "bounds": "([xmin,xmax],[ymin,ymax])", "resolution": 5.0}
```

---
## 3. Inspección de las nubes de puntos

### 3.1 Quickinfo — metadatos sin cargar datos

`pipeline.quickinfo` lee solo el **header** del archivo: es prácticamente instantáneo incluso con 300 M de puntos.

In [ ]:
def quickinfo(ruta):
    """Devuelve el quickinfo de un archivo LAS/LAZ/COPC."""
    reader_type = 'readers.copc' if str(ruta).endswith('.laz') else 'readers.las'
    p = pdal.Pipeline(json.dumps({'pipeline': [{'type': reader_type, 'filename': str(ruta)}]}))
    return p.quickinfo

# ── Inspeccionamos el L1 original ─────────────────────────────────────────────
qi_l1 = quickinfo(L1_ORIG)
b = qi_l1['readers.copc']['bounds']

print('=== L1 Zenmuse (original, sin CRS) ===')
print(f"  Puntos: {qi_l1['readers.copc']['num_points']:,}")
print(f"  X: [{b['minx']:.1f}, {b['maxx']:.1f}]")
print(f"  Y: [{b['miny']:.1f}, {b['maxy']:.1f}]")
print(f"  Z: [{b['minz']:.2f}, {b['maxz']:.2f}]")
print(f"  Dimensiones: {qi_l1['readers.copc']['dimensions']}")

In [ ]:
# ── Inspeccionamos el PNOA mosaico ────────────────────────────────────────────
qi_pnoa = quickinfo(PNOA_MOS)
b_pnoa = qi_pnoa['readers.copc']['bounds']

print('=== PNOA 2024 (mosaico 11 tiles) ===')
print(f"  Puntos: {qi_pnoa['readers.copc']['num_points']:,}")
print(f"  X: [{b_pnoa['minx']:.1f}, {b_pnoa['maxx']:.1f}]")
print(f"  Y: [{b_pnoa['miny']:.1f}, {b_pnoa['maxy']:.1f}]")
print(f"  Z: [{b_pnoa['minz']:.2f}, {b_pnoa['maxz']:.2f}]")

Ya podemos observar algo llamativo: el **rango Z del L1** (72–118 m) es muy diferente al del **PNOA** (17–64 m) aunque cubren el mismo terreno.  
Esto se explica en la **sección 5** (alturas elipsoidales vs ortométricas).

### 3.2 Metadatos completos: CRS y software

In [ ]:
def get_metadata(ruta, reader='readers.copc'):
    """Ejecuta el reader y extrae los metadatos completos."""
    p = pdal.Pipeline(json.dumps({'pipeline': [{'type': reader, 'filename': str(ruta)}]}))
    p.execute()
    md = p.metadata
    if isinstance(md, str):   # compatibilidad con versiones antiguas de PDAL
        md = json.loads(md)
    return md['metadata'][reader]

# Inspeccionamos el CRS del L1 original
md_l1 = get_metadata(L1_ORIG)
print('=== CRS del L1 original ===')
print('  comp_spatialreference:', md_l1.get('comp_spatialreference', '(vacío)'))
print('  spatialreference:',      md_l1.get('spatialreference', '(vacío)'))
print('  software_id:',           md_l1.get('software_id', '(vacío)')[:60])

In [ ]:
# Comparamos con el PNOA — que sí tiene CRS correcto
md_pnoa = get_metadata(PNOA_MOS)
print('=== CRS del PNOA ===')
srs = md_pnoa.get('srs', {})
if srs:
    print('  WKT:', srs.get('compoundwkt', '')[:120], '...')
else:
    # En algunas versiones está en otra clave
    print('  spatialreference:', md_pnoa.get('spatialreference', '(vacío)')[:120])

### 3.3 Estadísticas de clasificaciones

El **estándar LAS** define una serie de clases para los puntos:

| Código | Significado |
|--------|-------------|
| 0 | Nunca clasificado |
| 1 | Sin clasificar |
| 2 | Suelo |
| 3 | Vegetación baja |
| 4 | Vegetación media |
| 5 | Vegetación alta |
| 6 | Edificio |
| 7 | Ruido |
| 9 | Agua |
| 12 | Solapamiento |

El PNOA oficial está clasificado. El L1 del dron, dependiendo del procesado en DJI Terra, puede estar parcialmente clasificado (solo suelo y no clasificado).

In [ ]:
NOMBRES_CLASE = {
    0: 'Nunca clasificado',
    1: 'Sin clasificar',
    2: 'Suelo',
    3: 'Vegetación baja',
    4: 'Vegetación media',
    5: 'Vegetación alta',
    6: 'Edificio',
    7: 'Ruido',
    9: 'Agua',
    12: 'Solapamiento',
}

def stats_clasificacion(ruta, label, resolution=2.0):
    """
    Calcula la distribución de clasificaciones usando una muestra (resolution=2.0)
    para no cargar toda la nube cuando es muy grande.
    """
    pipeline = {
        'pipeline': [
            {'type': 'readers.copc', 'filename': str(ruta), 'resolution': resolution},
            {'type': 'filters.stats', 'dimensions': 'Classification', 'count': 'Classification'}
        ]
    }
    p = pdal.Pipeline(json.dumps(pipeline))
    p.execute()
    md = p.metadata
    if isinstance(md, str):
        md = json.loads(md)
    stat = md['metadata']['filters.stats']['statistic'][0]
    total = stat['count']

    print(f'\n=== Clasificaciones: {label} (muestra con resolución {resolution} m) ===')
    print(f'  Total puntos en muestra: {total:,}')
    for entry in stat.get('counts', []):
        clase_str, cnt_str = entry.split('/')
        clase = int(float(clase_str))
        cnt   = int(cnt_str)
        pct   = 100 * cnt / total
        nombre = NOMBRES_CLASE.get(clase, f'Clase {clase}')
        bar = '█' * int(pct / 2)
        print(f'  Clase {clase:2d} ({nombre:<20}): {cnt:>10,} pts  {pct:5.1f}%  {bar}')

stats_clasificacion(L1_ORIG, 'L1 Zenmuse (original)')
stats_clasificacion(PNOA_MOS, 'PNOA 2024')

---
## 4. Visualización de perfiles y distribución Z

Para visualizar usamos `laspy` directamente ya que carga los datos en arrays NumPy.  
Tomamos una **muestra aleatoria** para no saturar la RAM.

In [ ]:
def leer_muestra(ruta, n=100_000):
    """Lee n puntos aleatorios de un archivo LAS/LAZ."""
    las = laspy.read(str(ruta))
    total = len(las.points)
    idx = np.random.choice(total, min(n, total), replace=False)
    return las, idx

# Leer muestras de ambas nubes
print('Leyendo muestra del L1...')
las_l1, idx_l1 = leer_muestra(L1_ORTOM, n=100_000)
print(f'  {len(idx_l1):,} puntos')

print('Leyendo muestra del PNOA recortado...')
las_pnoa, idx_pnoa = leer_muestra(PNOA_CROP, n=100_000)
print(f'  {len(idx_pnoa):,} puntos')

In [ ]:
# Paleta de colores por clasificación
COLORES_CLASE = {
    1: '#aaaaaa',  # sin clasificar
    2: '#8B4513',  # suelo
    3: '#90EE90',  # veg baja
    4: '#228B22',  # veg media
    5: '#006400',  # veg alta
    6: '#FF0000',  # edificio
    12: '#FFD700', # solapamiento
}

def color_por_clase(clases):
    colores = np.full((len(clases), 3), 0.7)  # gris por defecto
    for cls, hex_color in COLORES_CLASE.items():
        rgb = mcolors.to_rgb(hex_color)
        colores[clases == cls] = rgb
    return colores

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Comparativa L1 Zenmuse vs PNOA 2024 — Alcolea del Río', fontsize=14, fontweight='bold')

datasets = [
    (las_l1, idx_l1, 'L1 Zenmuse (corregido)'),
    (las_pnoa, idx_pnoa, 'PNOA 2024 (recortado)')
]

for row, (las, idx, titulo) in enumerate(datasets):
    x = np.asarray(las.x)[idx]
    y = np.asarray(las.y)[idx]
    z = np.asarray(las.z)[idx]
    clases = np.asarray(las.classification)[idx]
    colores = color_por_clase(clases)

    # Vista en planta coloreada por altura
    ax = axes[row, 0]
    sc = ax.scatter(x, y, c=z, s=0.5, cmap='terrain', alpha=0.6)
    ax.set_title(f'{titulo}\nVista en planta (color = altura)')
    ax.set_aspect('equal')
    plt.colorbar(sc, ax=ax, label='Z (m)', fraction=0.046)

    # Perfil lateral (XZ) coloreado por clasificación
    ax = axes[row, 1]
    ax.scatter(x, z, c=colores, s=0.3, alpha=0.5)
    ax.set_title(f'{titulo}\nPerfil lateral (color = clase)')
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Z (m)')

    # Histograma de alturas
    ax = axes[row, 2]
    ax.hist(z, bins=100, alpha=0.7, color='steelblue', edgecolor='none')
    ax.set_title(f'{titulo}\nDistribución de alturas')
    ax.set_xlabel('Z (m)')
    ax.set_ylabel('Frecuencia')
    ax.axvline(np.median(z), color='red', linestyle='--', label=f'Mediana: {np.median(z):.1f} m')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. El problema del CRS en DJI Terra: alturas elipsoidales vs ortométricas

> ⚠️ Esta sección es **especialmente importante** para cualquiera que trabaje con drones DJI.

### El problema

El L1 original tiene Z ≈ 72–118 m, pero el PNOA tiene Z ≈ 17–64 m.  
Están en la misma zona. ¿Por qué difieren ~50 m en altura?

### Dos referencias de altura

```
        Superficie terrestre
             │
             │ H (altura ortométrica)    ← lo que queremos
             │   "altura sobre el mar"
         Geoide
             │
             │ N (ondulación del geoide)
             │   ~49.5 m en Alcolea del Río
        Elipsoide WGS84
             │
             │ h (altura elipsoidal)     ← lo que da el GPS
```

**Relación fundamental:**
```
H (ortométrica) = h (elipsoidal) - N (ondulación del geoide)
```

En Alcolea del Río, **N ≈ 49.5 m** (geoide EGM08-REDNAP, geoide oficial IGN España peninsular).

### ¿Por qué pasa esto en DJI?

DJI Terra exporta por defecto con **altura elipsoidal WGS84** (`EPSG:4979` en vertical)  
y **no escribe el CRS en el header del LAS** — es un bug conocido de Terra.

El header aparece con `software_id: "DJI TERRA ... unknown"` y todos los campos SRS vacíos.

### Solución: `override_srs` + `filters.reprojection` + geoide

PDAL puede forzar el CRS de entrada con `override_srs` y reproyectar aplicando el geoide oficial:

In [ ]:
# Verificamos que tenemos el geoide oficial de España instalado
# (requiere proj-data instalado vía conda)
import subprocess
resultado = subprocess.run(
    ['projinfo', '-s', 'EPSG:4979', '-t', 'EPSG:25830+5782'],
    capture_output=True, text=True
)
if 'es_ign_egm08-rednap' in resultado.stdout:
    print('✓ Geoide EGM08-REDNAP disponible')
    # Mostrar las primeras líneas con info del geoide
    for linea in resultado.stdout.split('\n')[:15]:
        print(linea)
else:
    print('✗ Geoide no encontrado. Instalar con: conda install -c conda-forge proj-data')
    print(resultado.stdout[:300])

In [ ]:
# ── SANITY CHECK: reproyección con muestra baja resolución ─────────────────────
# resolution=5.0 → lee solo el nivel de detalle 5 m del COPC (segundos, no minutos)

print('Sanity check reproyección (muestra con resolution=5.0)...')

sanity = pdal.Pipeline(json.dumps({
    'pipeline': [
        {
            'type': 'readers.copc',
            'filename': str(L1_ORIG),
            'override_srs': 'EPSG:32630+4979',   # forzamos el CRS que Terra no escribió
            'resolution': 5.0
        },
        {
            'type': 'filters.reprojection',
            'out_srs': 'EPSG:25830+5782'          # ETRS89/UTM30N + altura ortométrica Alicante
        }
    ]
}))
sanity.execute()
arr = sanity.arrays[0]

print(f'Puntos en muestra: {len(arr):,}')
print(f'Z antes (elipsoidal):  72.8 - 118.9 m  (conocido del header)')
print(f'Z después (ort.):      {arr["Z"].min():.2f} - {arr["Z"].max():.2f} m')
print(f'Z media:               {arr["Z"].mean():.2f} m')
print()
print('¿Los valores tienen sentido para Alcolea del Río (~20-70 m s.n.m.)? ', end='')
if 15 < arr['Z'].mean() < 80:
    print('✓ Sí')
else:
    print('✗ Revisar el geoide')

In [ ]:
# ── PIPELINE COMPLETO DE REPROYECCIÓN ─────────────────────────────────────────
# ⚠️  TIEMPO: ~10-15 minutos para 303 M puntos (proceso monohilo)
# ⚠️  NO RE-EJECUTAR si l1_ortometrica.copc.laz ya existe

if L1_ORTOM.exists():
    print(f'✓ {L1_ORTOM.name} ya existe. No es necesario reproyectar.')
    qi = pdal.Pipeline(json.dumps({'pipeline': [{
        'type': 'readers.copc', 'filename': str(L1_ORTOM)
    }]})).quickinfo
    b = qi['readers.copc']['bounds']
    print(f'  Puntos: {qi["readers.copc"]["num_points"]:,}')
    print(f'  Z: [{b["minz"]:.2f}, {b["maxz"]:.2f}] m  ← alturas ortométricas ✓')
else:
    print('Ejecutando reproyección completa...')
    t0 = time.time()

    pipeline_reproyeccion = {
        'pipeline': [
            {
                'type': 'readers.copc',
                'filename': str(L1_ORIG),
                'override_srs': 'EPSG:32630+4979'   # WGS84/UTM30N + elipsoidal
            },
            {
                'type': 'filters.reprojection',
                'out_srs': 'EPSG:25830+5782'         # ETRS89/UTM30N + ortométrica
            },
            {
                'type': 'writers.copc',              # COPC preserva el índice octree
                'filename': str(L1_ORTOM)
            }
        ]
    }

    count = pdal.Pipeline(json.dumps(pipeline_reproyeccion)).execute()
    print(f'Procesados {count:,} puntos en {(time.time()-t0)/60:.1f} min')

---
## 6. Recorte del PNOA al área de vuelo

El PNOA cubre una zona mucho mayor que el vuelo del L1. Para la comparación necesitamos recortarlo al mismo extent.

### Dos estrategias

1. **Bounding box**: rápido pero incluye esquinas vacías si el vuelo no fue rectangular
2. **Hull hexbin**: calcula el contorno real del vuelo y recorta ajustado

Usaremos las dos encadenadas: primero el bbox (descarta el 95% del PNOA rápidamente), luego el polígono preciso.

In [ ]:
# ── Paso 1: Leer los bounds del L1 ortométrico ────────────────────────────────
qi_l1_ort = pdal.Pipeline(json.dumps({
    'pipeline': [{'type': 'readers.copc', 'filename': str(L1_ORTOM)}]
})).quickinfo

b = qi_l1_ort['readers.copc']['bounds']
print(f'Extent L1 ortométrico:')
print(f'  X: [{b["minx"]:.1f}, {b["maxx"]:.1f}]')
print(f'  Y: [{b["miny"]:.1f}, {b["maxy"]:.1f}]')

# Buffer de seguridad
BUFFER = 20
bbox_str = f'([{b["minx"]-BUFFER:.1f},{b["maxx"]+BUFFER:.1f}],[{b["miny"]-BUFFER:.1f},{b["maxy"]+BUFFER:.1f}])'
print(f'\nBbox con buffer {BUFFER} m: {bbox_str}')

In [ ]:
# ── Paso 2: Recortar PNOA por bounding box ────────────────────────────────────
if PNOA_CROP.exists():
    qi_crop = pdal.Pipeline(json.dumps({
        'pipeline': [{'type': 'readers.copc', 'filename': str(PNOA_CROP)}]
    })).quickinfo
    print(f'✓ {PNOA_CROP.name} ya existe.')
    print(f'  Puntos: {qi_crop["readers.copc"]["num_points"]:,}')
else:
    print('Recortando PNOA por bbox...')
    crop_bbox = {
        'pipeline': [
            {'type': 'readers.copc', 'filename': str(PNOA_MOS)},
            {'type': 'filters.crop', 'bounds': bbox_str},
            {'type': 'writers.copc', 'filename': str(PNOA_CROP)}
        ]
    }
    count = pdal.Pipeline(json.dumps(crop_bbox)).execute()
    print(f'PNOA recortado: {count:,} puntos')

In [ ]:
# ── Calcular el hull real del vuelo L1 con filters.hexbin ─────────────────────
# resolution=2.0 para no cargar toda la nube (tarda segundos)

print('Calculando hull hexbin del L1...')

hull_pipeline = {
    'pipeline': [
        {
            'type': 'readers.copc',
            'filename': str(L1_ORTOM),
            'resolution': 2.0
        },
        {
            'type': 'filters.hexbin',
            'edge_size': 5.0,    # hexágonos de 5 m → buen equilibrio detalle/velocidad
            'threshold': 1       # mínimo 1 punto por hexágono
        }
    ]
}

p = pdal.Pipeline(json.dumps(hull_pipeline))
p.execute()

md = p.metadata
if isinstance(md, str):
    md = json.loads(md)
hull_wkt = md['metadata']['filters.hexbin']['boundary']

print(f'Hull calculado ({len(hull_wkt):,} caracteres WKT)')
print(f'Inicio del WKT: {hull_wkt[:150]}...')

In [ ]:
# Visualizar el hull sobre el extent del PNOA
from shapely import wkt as shapely_wkt

hull_geom = shapely_wkt.loads(hull_wkt)
gdf_hull = gpd.GeoDataFrame(geometry=[hull_geom], crs='EPSG:25830')

fig, ax = plt.subplots(figsize=(8, 10))
gdf_hull.plot(ax=ax, facecolor='#4A90D9', edgecolor='#1A5276', alpha=0.5, linewidth=1.5)
ax.set_title('Contorno real del vuelo L1\n(hull hexbin de 5 m)', fontsize=12)
ax.set_xlabel('X ETRS89/UTM30N (m)')
ax.set_ylabel('Y ETRS89/UTM30N (m)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# Guardar como GeoJSON para QGIS
hull_path = WORKDIR / 'l1_hull.geojson'
gdf_hull.to_file(str(hull_path), driver='GeoJSON')
print(f'Hull guardado en: {hull_path}')

---
## 7. Generación de DTM, DSM y CHM

### Conceptos clave

```
  ┌───────────────────────────────────────────────────────────┐
  │                                                           │
  │    ●  ●                           ●  ● ← DSM (superficie)│
  │  ●      ●     ●  ●  ●  ●  ●  ●  ●      (máx. Z)         │
  │                                                    CHM   │
  │──────────────────────────────────────── ← DTM (terreno)  │
  │                  (clase 2, interpolado)                   │
  └───────────────────────────────────────────────────────────┘
                  CHM = DSM - DTM
```

| Modelo | Qué representa | Puntos usados | Interpolación |
|--------|----------------|---------------|---------------|
| **DTM** | Terreno desnudo | Solo clase 2 (suelo) | IDW (inverso distancia) |
| **DSM** | Superficie superior | Primeros retornos (ReturnNumber=1) | Máximo Z |
| **CHM** | Altura de vegetación | — | DSM − DTM |

### Interpolación IDW para el DTM

Los puntos de suelo (clase 2) no cubren uniformemente el terreno: bajo las copas de los árboles hay huecos. Usamos **IDW** (_Inverse Distance Weighting_) con `window_size=5` para rellenar esos huecos de forma razonable.

### Bounds fijos para alineación píxel a píxel

Para que los rasters del L1 y el PNOA tengan exactamente la misma cuadrícula (requisito para la resta CHM y la comparación), definimos bounds con valores enteros (alineados al metro).

In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
RES    = 1.0  # resolución en metros
BOUNDS = '([267456,267954],[4165567,4166888])'  # bounds enteros para alineación

# ── Funciones generadoras ─────────────────────────────────────────────────────

def generar_dtm(input_laz, output_tif, res=RES, bounds=BOUNDS):
    """
    DTM: solo puntos de suelo (clase 2), interpolación IDW.
    IDW rellena huecos bajo la vegetación de forma suave.
    """
    pipeline = {
        'pipeline': [
            {'type': 'readers.copc', 'filename': str(input_laz)},
            {'type': 'filters.range', 'limits': 'Classification[2:2]'},
            {
                'type': 'writers.gdal',
                'filename': str(output_tif),
                'resolution': res,
                'bounds': bounds,
                'output_type': 'idw',
                'window_size': 5,
                'nodata': -9999
            }
        ]
    }
    pdal.Pipeline(json.dumps(pipeline)).execute()
    print(f'  DTM → {Path(output_tif).name}')


def generar_dsm(input_laz, output_tif, res=RES, bounds=BOUNDS):
    """
    DSM: primeros retornos, valor máximo de Z por píxel.
    Los primeros retornos representan la superficie visible desde el sensor.
    """
    pipeline = {
        'pipeline': [
            {'type': 'readers.copc', 'filename': str(input_laz)},
            {'type': 'filters.range', 'limits': 'ReturnNumber[1:1]'},
            {
                'type': 'writers.gdal',
                'filename': str(output_tif),
                'resolution': res,
                'bounds': bounds,
                'output_type': 'max',
                'window_size': 3,
                'nodata': -9999
            }
        ]
    }
    pdal.Pipeline(json.dumps(pipeline)).execute()
    print(f'  DSM → {Path(output_tif).name}')


def generar_chm(dsm_path, dtm_path, output_path):
    """
    CHM = DSM - DTM. Operación raster con rasterio.
    Clampeamos CHM < 0 a 0 (no hay vegetación negativa).
    Eliminamos outliers > 50 m.
    """
    with rasterio.open(dsm_path) as dsm_src, rasterio.open(dtm_path) as dtm_src:
        dsm     = dsm_src.read(1).astype(np.float32)
        dtm     = dtm_src.read(1).astype(np.float32)
        profile = dsm_src.profile
        nd_dsm  = dsm_src.nodata
        nd_dtm  = dtm_src.nodata

    nodata = -9999.0
    mask   = (dsm == nd_dsm) | (dtm == nd_dtm)
    chm    = dsm - dtm
    chm[mask]              = nodata
    chm[(~mask) & (chm < 0)]  = 0
    chm[(~mask) & (chm > 50)] = nodata   # outliers probables

    profile.update(dtype=rasterio.float32, nodata=nodata, compress='lzw')
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(chm, 1)
    print(f'  CHM → {Path(output_path).name}')


print('Funciones definidas.')

In [ ]:
# ── Generar DTM/DSM/CHM para el L1 ───────────────────────────────────────────
# Tiempo estimado: 1-3 min por raster

if all(p.exists() for p in [DTM_L1, DSM_L1, CHM_L1]):
    print('✓ Rasters del L1 ya existen, saltando generación.')
else:
    print('Generando rasters del L1...')
    t0 = time.time()
    generar_dtm(L1_ORTOM, DTM_L1)
    generar_dsm(L1_ORTOM, DSM_L1)
    generar_chm(DSM_L1, DTM_L1, CHM_L1)
    print(f'  Tiempo L1: {time.time()-t0:.0f} s')

# ── Generar DTM/DSM/CHM para el PNOA ─────────────────────────────────────────
if all(p.exists() for p in [DTM_PNOA, DSM_PNOA, CHM_PNOA]):
    print('✓ Rasters del PNOA ya existen, saltando generación.')
else:
    print('Generando rasters del PNOA...')
    t0 = time.time()
    generar_dtm(PNOA_CROP, DTM_PNOA)
    generar_dsm(PNOA_CROP, DSM_PNOA)
    generar_chm(DSM_PNOA, DTM_PNOA, CHM_PNOA)
    print(f'  Tiempo PNOA: {time.time()-t0:.0f} s')

print('\n✓ Los 6 rasters están listos.')

---
## 8. Comparación visual: los 6 rasters

Visualizamos los 3 modelos (DTM, DSM, CHM) de cada fuente en una cuadrícula 2×3.

In [ ]:
def leer_raster(path):
    """Lee un raster y devuelve el array con nodata enmascarado."""
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float32)
        nd = src.nodata
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    if nd is not None:
        data = np.where(data == nd, np.nan, data)
    return data, extent

# Leer los 6 rasters
dtm_l1_arr,  ext = leer_raster(DTM_L1)
dsm_l1_arr,  _   = leer_raster(DSM_L1)
chm_l1_arr,  _   = leer_raster(CHM_L1)
dtm_pnoa_arr, _  = leer_raster(DTM_PNOA)
dsm_pnoa_arr, _  = leer_raster(DSM_PNOA)
chm_pnoa_arr, _  = leer_raster(CHM_PNOA)

# Estadísticas básicas
for nombre, arr in [('DTM L1',  dtm_l1_arr),  ('DSM L1',  dsm_l1_arr),  ('CHM L1',  chm_l1_arr),
                    ('DTM PNOA', dtm_pnoa_arr), ('DSM PNOA', dsm_pnoa_arr), ('CHM PNOA', chm_pnoa_arr)]:
    v = arr[~np.isnan(arr)]
    if len(v):
        print(f'  {nombre:<12}: min={v.min():.2f}  max={v.max():.2f}  media={v.mean():.2f}  nodata={np.isnan(arr).mean()*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('DTM / DSM / CHM — L1 Zenmuse vs PNOA 2024', fontsize=15, fontweight='bold')

config = [
    # (array,         título,              colormap,  vmin, vmax)
    (dtm_l1_arr,   'DTM L1\n(1 m/píxel)',  'terrain',    15,  70),
    (dsm_l1_arr,   'DSM L1\n(1 m/píxel)',  'terrain',    15,  90),
    (chm_l1_arr,   'CHM L1\n(vegetación)', 'YlGn',        0,  25),
    (dtm_pnoa_arr, 'DTM PNOA\n(1 m/píxel)','terrain',    15,  70),
    (dsm_pnoa_arr, 'DSM PNOA\n(1 m/píxel)','terrain',    15,  90),
    (chm_pnoa_arr, 'CHM PNOA\n(vegetación)','YlGn',       0,  25),
]

for ax, (arr, titulo, cmap, vmin, vmax) in zip(axes.flat, config):
    im = ax.imshow(arr, extent=ext, cmap=cmap, vmin=vmin, vmax=vmax,
                   interpolation='nearest', aspect='auto')
    ax.set_title(titulo, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, label='m')
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(str(WORKDIR / 'comparativa_dtm_dsm_chm.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Cuantificación del sesgo Z: base autoposicionada

### El problema

Tras corregir la ondulación del geoide (~49.5 m), puede persistir un **sesgo sistemático** de 2–5 m entre el L1 y el PNOA.

**Causa**: la base D-RTK 2 fue dejada en **modo autoposicionamiento** (modo A).  
En este modo la base se posiciona por GNSS autónomo con error absoluto típico de:
- **Vertical**: 2–5 m
- **Horizontal**: 1–3 m

La precisión **relativa** interna de la nube (entre puntos) sigue siendo centimétrica.  
Solo la exactitud **absoluta** (respecto a coordenadas reales) está degradada.

### Estrategia de corrección

1. Calcular la diferencia `DTM_L1 - DTM_PNOA` solo en **zonas de suelo desnudo** (caminos, eras) donde la comparación es directa
2. Calcular la **mediana** del sesgo (robusta frente a outliers)
3. Aplicar el offset constante a toda la nube L1

In [ ]:
# ── Calcular diferencia DTM_L1 - DTM_PNOA ────────────────────────────────────

# Máscara de píxeles válidos en ambos rasters
mask_valido = ~np.isnan(dtm_l1_arr) & ~np.isnan(dtm_pnoa_arr)

diff_z = dtm_l1_arr - dtm_pnoa_arr
diff_valido = diff_z[mask_valido]

sesgo_mediana = np.median(diff_valido)
sesgo_media   = np.mean(diff_valido)
sesgo_std     = np.std(diff_valido)

print(f'Diferencia DTM_L1 - DTM_PNOA en píxeles con datos válidos:')
print(f'  N píxeles:  {mask_valido.sum():,}')
print(f'  Mediana:    {sesgo_mediana:+.3f} m  ← offset recomendado')
print(f'  Media:      {sesgo_media:+.3f} m')
print(f'  Desv. std:  {sesgo_std:.3f} m')
print()
if abs(sesgo_mediana) < 0.5:
    print('✓ Sesgo < 0.5 m — la corrección de geoide fue suficiente')
elif abs(sesgo_mediana) < 3:
    print(f'⚠ Sesgo de {sesgo_mediana:.2f} m — típico de base autoposicionada')
else:
    print(f'⚠ Sesgo grande ({sesgo_mediana:.2f} m) — verificar corrección de geoide')

In [ ]:
# ── Visualización del sesgo ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Análisis del sesgo Z: L1 Zenmuse vs PNOA 2024', fontsize=13, fontweight='bold')

# Mapa de diferencias
ax = axes[0]
diff_display = np.where(mask_valido, diff_z, np.nan)
vmax_diff = max(abs(np.nanpercentile(diff_display, 2)), abs(np.nanpercentile(diff_display, 98)))
im = ax.imshow(diff_display, extent=ext, cmap='RdBu_r',
               vmin=-vmax_diff, vmax=vmax_diff, aspect='auto', interpolation='nearest')
ax.set_title('Mapa de diferencias DTM L1 − DTM PNOA\n(rojo = L1 más alto, azul = L1 más bajo)')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.tick_params(axis='x', rotation=45)
plt.colorbar(im, ax=ax, label='ΔZ (m)')

# Histograma del sesgo
ax = axes[1]
ax.hist(diff_valido, bins=100, alpha=0.7, color='steelblue', edgecolor='none', density=True)
ax.axvline(sesgo_mediana, color='red',    linewidth=2, label=f'Mediana: {sesgo_mediana:+.3f} m')
ax.axvline(sesgo_media,   color='orange', linewidth=2, linestyle='--', label=f'Media: {sesgo_media:+.3f} m')
ax.axvline(0,             color='black',  linewidth=1, linestyle=':')
ax.set_title('Distribución del sesgo Z')
ax.set_xlabel('ΔZ = Z_L1 − Z_PNOA (m)')
ax.set_ylabel('Densidad')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(WORKDIR / 'sesgo_z_l1_vs_pnoa.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Corrección del sesgo con PDAL ─────────────────────────────────────────────
# filters.assign permite modificar dimensiones de forma algebraica

L1_ALINEADA = WORKDIR / 'l1_alineada.copc.laz'

if abs(sesgo_mediana) < 0.3:
    print(f'Sesgo ({sesgo_mediana:.3f} m) menor que la precisión del PNOA → no se aplica corrección.')
elif L1_ALINEADA.exists():
    print(f'✓ {L1_ALINEADA.name} ya existe.')
else:
    print(f'Aplicando offset Z de {-sesgo_mediana:.3f} m al L1...')

    correccion = {
        'pipeline': [
            {'type': 'readers.copc', 'filename': str(L1_ORTOM)},
            {'type': 'filters.assign', 'value': f'Z = Z - {sesgo_mediana:.4f}'},
            {'type': 'writers.copc',  'filename': str(L1_ALINEADA)}
        ]
    }

    t0 = time.time()
    count = pdal.Pipeline(json.dumps(correccion)).execute()
    print(f'  {count:,} puntos procesados en {(time.time()-t0)/60:.1f} min')
    print(f'  Archivo: {L1_ALINEADA}')

print()
print('Nota sobre precisiones del PNOA-LiDAR:')
print('  - Precisión altimétrica PNOA 3ª cobertura: ≤ 0.20 m RMSE')
print('  - Precisión planimétrica: ≤ 0.30 m RMSE')
print('  → Correcciones < 0.3 m están dentro del ruido del propio PNOA')

---
## 10. Comparación de CHMs: L1 vs PNOA

Con el sesgo Z corregido, podemos comparar los **modelos de altura de vegetación** de ambas fuentes.  
Diferencias en el CHM reflejan:
- **Cambios reales de vegetación** entre las fechas
- **Diferencias de densidad de puntos** (el L1 es ~50× más denso que el PNOA)
- **Diferencias de clasificación** (el PNOA tiene clasificación automática estandarizada)

In [ ]:
# Aplicar la corrección de sesgo al CHM L1 si existe
chm_l1_corr = chm_l1_arr - sesgo_mediana if abs(sesgo_mediana) > 0.3 else chm_l1_arr

# Diferencia CHM L1 - CHM PNOA
mask_chm = ~np.isnan(chm_l1_corr) & ~np.isnan(chm_pnoa_arr)
diff_chm = chm_l1_corr - chm_pnoa_arr
diff_chm_valido = diff_chm[mask_chm]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Comparación CHM: L1 Zenmuse vs PNOA 2024', fontsize=13, fontweight='bold')

# CHM L1
im1 = axes[0].imshow(chm_l1_corr, extent=ext, cmap='YlGn', vmin=0, vmax=20, aspect='auto')
axes[0].set_title('CHM L1 Zenmuse')
plt.colorbar(im1, ax=axes[0], label='Altura veg. (m)')

# CHM PNOA
im2 = axes[1].imshow(chm_pnoa_arr, extent=ext, cmap='YlGn', vmin=0, vmax=20, aspect='auto')
axes[1].set_title('CHM PNOA 2024')
plt.colorbar(im2, ax=axes[1], label='Altura veg. (m)')

# Diferencia
diff_chm_display = np.where(mask_chm, diff_chm, np.nan)
vmax_c = np.nanpercentile(np.abs(diff_chm_display), 95)
im3 = axes[2].imshow(diff_chm_display, extent=ext, cmap='RdBu_r',
                     vmin=-vmax_c, vmax=vmax_c, aspect='auto')
axes[2].set_title('Diferencia CHM\n(L1 − PNOA)')
plt.colorbar(im3, ax=axes[2], label='ΔCHM (m)')

for ax in axes:
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(str(WORKDIR / 'comparativa_chm.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\nDiferencia CHM (L1 - PNOA):')
print(f'  Mediana: {np.median(diff_chm_valido):+.2f} m')
print(f'  RMSE:    {np.sqrt(np.mean(diff_chm_valido**2)):.2f} m')
print(f'  Rango IQR (P25-P75): [{np.percentile(diff_chm_valido,25):.2f}, {np.percentile(diff_chm_valido,75):.2f}] m')

---
## Resumen de conceptos clave

### PDAL
- **Pipeline JSON**: lectura → filtros → escritura. Reproducible y documentable.
- **`readers.copc` con `resolution`**: lectura parcial para exploración rápida (sanity checks en segundos).
- **`filters.range`**: filtrar por cualquier dimensión (clase, número de retorno, intensidad…).
- **`filters.crop`**: recorte por bbox o polígono WKT.
- **`filters.hexbin`**: calcula el hull real de la nube.
- **`writers.gdal`**: rasterización directa a GeoTIFF (DTM, DSM).
- **`filters.assign`**: modificación algebraica de dimensiones (ej: corrección Z).

### Alturas elipsoidales vs ortométricas
- DJI Terra exporta en altura **elipsoidal WGS84** y no escribe el CRS en el LAS.
- Corrección: `override_srs=EPSG:32630+4979` + `filters.reprojection` → `EPSG:25830+5782`.
- Necesita `proj-data` con el geoide `es_ign_egm08-rednap.tif`.

### Sesgo de base autoposicionada
- Base D-RTK 2 en modo A → precisión relativa centimétrica, exactitud absoluta ±2–5 m.
- Corrección: mediana de `DTM_L1 − DTM_PNOA` en zonas de suelo desnudo + `filters.assign`.

### CHM
- CHM = DSM − DTM. Requiere que ambos rasters tengan el mismo grid.
- Diferencias L1 vs PNOA: densidad de puntos, clasificación automática, y cambios temporales reales.